# Deploy a scikit-learn model on TrueFoundry with FastAPI, no inference code

- Flow: **train → log model → package → build & push image → deploy → infer**
- No inference code is written by hand — the runtime is generated or supplied
- Here we own the image: build locally, push to Docker Hub, deploy that tag. TrueFoundry never builds.
- Docs: https://www.truefoundry.com/docs/model-deployment/deploy-sklearn-xgboost

---

## 0. Setup

> **Kernel: `Python 3`** — the default env (Python 3.14).

In [ ]:
# %pip install -U truefoundry scikit-learn joblib requests

In [ ]:
import sys
import time
from pathlib import Path

import joblib
import requests
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier

from truefoundry.ml import get_client, SklearnFramework, sklearn_infer_schema
from truefoundry.common.session import Session
from truefoundry_sdk import TrueFoundry

In [ ]:
TFY_HOST = "https://internal.devtest.truefoundry.tech"
ML_REPO = "default"
CLUSTER = "tfy-usea1-devtest"
WORKSPACE = "seldon-tf"
DOMAIN = "tfy-usea1-ctl.devtest.truefoundry.tech"
WS_FQN = f"{CLUSTER}:{WORKSPACE}"

DOCKER_USER = "mkozubal"
TAG = "1"
PY_VER = "3.14"
PORT = 8000
READY_TIMEOUT = 300

NAME = "iris-sk-fastapi"
MODEL_FILE = f"{NAME}.joblib"
SVC = f"{NAME}-svc"
IMAGE = f"{DOCKER_USER}/{SVC}:{TAG}"
HOST = f"{SVC}-{WORKSPACE}-{PORT}.{DOMAIN}"

## 1. Train and save

In [ ]:
X, y = load_iris(return_X_y=True)
model = RandomForestClassifier(n_estimators=10).fit(X, y)
joblib.dump(model, MODEL_FILE)

## 2. Log model — schema goes here

`model_filepath` + `serialization_format` + `model_schema` are what make the version deployable.

In [ ]:
client = get_client()

framework = SklearnFramework(
    model_filepath=MODEL_FILE,
    serialization_format="joblib",
    model_schema=sklearn_infer_schema(
        model_input=X, model=model, infer_method_name="predict",
    ),
)

FQN = client.log_model(
    ml_repo=ML_REPO, name=NAME, model_file_or_folder=MODEL_FILE, framework=framework,
).fqn
print(FQN)

## 3. Generate the serving package

`tfy deploy-init` writes `server.py`, `infer.py`, `requirements.txt`, `deploy.py`, `README.md` locally.
Nothing is built and nothing is deployed — same command the UI Deploy button hands you.

Called as `{sys.executable} -m truefoundry.cli`: `!` inherits the notebook process PATH, which
does not contain the kernel venv's `bin/` unless Jupyter was started from an activated env.

In [ ]:
!{sys.executable} -m truefoundry.cli deploy-init model --name {SVC} --model-version-fqn '{FQN}' --workspace-fqn '{WS_FQN}' --model-server fastapi
assert _exit_code == 0, "deploy-init failed"

In [ ]:
assert Path(SVC).is_dir(), "deploy-init produced nothing"
print(sorted(p.name for p in Path(SVC).iterdir()))

## 4. Build and push the image

The FastAPI codegen emits **no** Dockerfile (its `deploy.py` uses `PythonBuild`, built on
TrueFoundry's builder). We add our own and push to Docker Hub instead — `deploy.py` goes unused.

In [ ]:
Path(SVC, "Dockerfile").write_text(f'''FROM python:{PY_VER}-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
CMD ["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "{PORT}"]
''')

In [ ]:
!docker build --platform linux/amd64 -t {IMAGE} {SVC}
assert _exit_code == 0, "docker build failed"

!docker push {IMAGE}
assert _exit_code == 0, "docker push failed"

## 5. Deploy as a service

Prebuilt image, so `"image": {"type": "image", ...}` instead of a build.
`artifacts_download` pulls the model version into `$MODEL_PATH` — the image stays model-free.

Auth reuses the `tfy login` session, so no PAT in the notebook.

`create_or_update` returns a **deployment**, so delete later with `app.data.application_id`,
not `app.data.id`.

In [ ]:
tfy = TrueFoundry(base_url=TFY_HOST, api_key=Session.new().access_token)

app = tfy.applications.create_or_update(manifest={
    "name": SVC, "type": "service", "workspace_fqn": WS_FQN,
    "image": {"type": "image", "image_uri": IMAGE},
    "artifacts_download": {"artifacts": [{"type": "truefoundry-artifact",
        "artifact_version_fqn": FQN, "download_path_env_variable": "MODEL_PATH"}]},
    "ports": [{"port": PORT, "protocol": "TCP", "expose": True,
               "app_protocol": "http", "host": HOST}],
    "resources": {"cpu_request": 0.5, "cpu_limit": 0.5,
                  "memory_request": 1000, "memory_limit": 1000,
                  "ephemeral_storage_request": 2000, "ephemeral_storage_limit": 2000},
    "replicas": 1,
})
print(SVC, app.data.application_id)

## 6. Call the model

URL is the `host` we set in `ports` above. Swagger at `https://<host>/`.
Poll `/health` first — the pod has to pull the image and download the model into `$MODEL_PATH`.

In [ ]:
deadline = time.time() + READY_TIMEOUT
while time.time() < deadline:
    try:
        if requests.get(f"https://{HOST}/health", timeout=5).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    raise TimeoutError(f"{SVC} not ready after {READY_TIMEOUT}s")

print(SVC, "ready")

In [ ]:
r = requests.post(f"https://{HOST}/infer",
                  json={"input_0": [[5.1, 3.5, 1.4, 0.2]]}, timeout=30)
print(r.status_code, r.text)

The same call as a copy-pasteable `curl`, for testing outside the notebook.

In [ ]:
print(f'''curl -X POST https://{HOST}/infer \\
  -H 'Content-Type: application/json' \\
  -d '{{"input_0": [[5.1, 3.5, 1.4, 0.2]]}}' ''')

## Cleanup

Commented out so a re-run does not wipe a service you still want. Uncomment to remove the
service, the model version, the local folder and the local image.
Unlinking the artifact is async — if the model-version delete says it is still in use, re-run.

In [ ]:
# tfy.applications.delete(id=app.data.application_id)
# client.get_model_version_by_fqn(FQN).delete()

# !rm -rf {SVC} {MODEL_FILE}
# !docker rmi -f {IMAGE}